# Chain Rule

Built on [Derivatives](derivatives.ipynb)

The Chain Rule is a calculus formula used to find the derivative of a composite function (a function inside another function), **it calculates the change of a Chain reaction**.

If a variable $z$ depends on $y$, and $y$ in turn depends on $x$ (i.e., $z = f(y)$ and $y = g(x))$, the Chain Rule states that the derivative of $z$ with respect to $x$ is the product of the derivative of $z$ with respect to $y$ and the derivative of $y$ with respect to $x$:

$$\frac{dz}{dx} = \frac{dz}{dy} * \frac{dy}{dx}$$

- Formula explanation:
  - We have two functions: $z=f(y)$ and $y=g(x)$.

  - $\frac{dy}{dx}$ Is the derivative of $y$ with respect to $x$. "How much does $y$ change when $x$ changes".

  - $\frac{dz}{dy}$ "How much does $z$ change when $y$ changes".

  - $\frac{dz}{dx}$ The total change in $z$ caused by $x$ is the product of those two individual changes.



## Chain Rule In Neural Networks

In deep learning, a neural network is a chain of composite functions, each layer's output is the input to the next layer. So a small change in the first layer's **weights** and **biases** causes a change in the output of that layer, which in turn causes a change in the next layer, and so on, until the final **Loss** error.

The Chain Rule is the mathematical backbone of **Backpropagation**. It efficiently calculates the **gradient** (derivative) of the final Loss error with respect to every single weight and bias in every layer of the network, i.e., how much each weight and bias contributed to that error. *We use those gradients to update the weights and biases so that the Loss error is lowered.*

A neural network isn't just two functions; it's a long chain of them. The logic is identical, you just keep multiplying.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

<p align="center">
    <img src="./showcase_images/Chain-Rule-notebook-neural-network.drawio.png" alt="Network Graph" width="95%">
</p>

**Network's Goal**: Given a number, is that number positive or negative.

In [2]:
class Net(nn.Module):
    """
    Simple network that predicts whether a number is a negative or positive. If it outputs a 0, it's predicting a negative, else 1 for positive. The network has two layers.
    """
    def __init__(self):
        super(Net, self).__init__()
        # Layer 1: Input (1 feature: the number ) -> 3 weight neurons
        self.fc1 = nn.Linear(1, 3)
        # Layer 2: Input (3 features) -> Output: negative or positive | 0 or 1
        self.fc2 = nn.Linear(3, 1)

    def forward(self, x):  # Forward pass
        z1 = self.fc1(x) # Pre-activation for layer 1
        a1 = F.relu(z1) # First layer activation
        z2 = self.fc2(a1) # Pre-activation for layer 2
        a2 = F.sigmoid(z2) # Model's final prediction
        y_hat = a2 # a2 and y_hat are just different notations for this, so we will only use y_hat
        return y_hat, z2, a1, z1
model = Net()

Input Data:

In [3]:
input_num = torch.tensor([[0.5]]) # The number the model will predict if it's negative or positive.
ground_truth = 1 if input_num.item() >= 0 else 0
ground_truth = torch.tensor([[ground_truth]], dtype=torch.float32)
classes = ["Negative", "Positive"]

print(f"Input number: {input_num.item():.4f} \n\nground_truth: the number is a {classes[int(ground_truth)]}.")

Input number: 0.5000 

ground_truth: the number is a Positive.


Force Model Weights:

- For this tutorial **only**, we will manually set the weights and biases to known values, this will make it easier to verify our calculations later on.

In [4]:
with torch.no_grad():
    model.fc1.weight.data = torch.tensor([[0.9], [0.7], [-0.3]])
    model.fc1.bias.data = torch.tensor([0.1, 0.2, 0.3])

    model.fc2.weight.data = torch.tensor([[0.3, 0.6, 0.5]])
    model.fc2.bias.data = torch.tensor([0.1])

Model prediction:

In [5]:
y_hat, z2, a1, z1 = model(input_num)
prediction = y_hat
print(f"Model Prediction: {round(float(prediction.item()))}\n")
print("If the prediction is a 0 then the model predicts that the integer is a negative, else 1 for a positive.")

Model Prediction: 1

If the prediction is a 0 then the model predicts that the integer is a negative, else 1 for a positive.


Calculate The Loss:

In [6]:
criterion = nn.BCELoss()
loss = criterion(prediction, ground_truth)
print(f"Loss: {loss.item():.4f}")

Loss: 0.4132


Note that the model hasn't been properly trained yet so its predictions are just random, its loss is likely to be terrible, i.e., not close to zero.

In [7]:
print(f"Input (X): {input_num.item():.4f}")
print(f"Ground truth: {ground_truth.item():.4f} | is a {classes[int(ground_truth)]}")
print("z1 (pre-ReLU):", z1.data)
print("a1 (post-ReLU):", a1.data)
print("z2 (pre-Sigmoid):", z2.data)
print(f"yhat (Prediction): {y_hat.item():.4f}")
print(f"Loss (L): {loss.item():.4f}")

Input (X): 0.5000
Ground truth: 1.0000 | is a Positive
z1 (pre-ReLU): tensor([[0.5500, 0.5500, 0.1500]])
a1 (post-ReLU): tensor([[0.5500, 0.5500, 0.1500]])
z2 (pre-Sigmoid): tensor([[0.6700]])
yhat (Prediction): 0.6615
Loss (L): 0.4132


### The Neural Network Chain

**Goal**: Find how much the weight node $W^{[1]}_1$ of the first layer contributes to the Loss $(L)$ error.

The path from $W^{[1]}_1$ to the Loss $L$ is:
- $W^{[1]}_1$ → $z^{[1]}_1$ → $a^{[1]}_1$ → $z^{[2]}$ → $\hat{y}$ → Loss $L$



To find the effect of $W^{[1]}_1$ on $L$ (which is $\frac{\partial L}{\partial W^{[1]}_1}$), we multiply all the local derivatives backward along the chain:

$$\frac{\partial L}{\partial W^{[1]}_1} = \frac{\partial L}{\partial \hat{y}} * \frac{\partial \hat{y}}{\partial z^{[2]}} * \frac{\partial z^{[2]}}{\partial a^{[1]}_1} * \frac{\partial a^{[1]}_1}{\partial z^{[1]}_1} * \frac{\partial z^{[1]}_1}{\partial W^{[1]}_1}$$

💡 **Note**: $\hat y$ and $a^{[2]}$ are the same thing, just different notations, so we will only use $\hat y$. That's why there's no $\frac{\partial \hat{y}}{\partial a^{[2]}}$ in the chain!


Formula explanation:
- $\frac{\partial L}{\partial W^{[1]}_1}$ is the Loss $L$ with respect to the first weight node of the first layer $W^{[1]}_1$
  
- $\frac{\partial L}{\partial \hat{y}}$ is the Loss $L$ with respect to the model's prediction $\hat y$

- $\frac{\partial \hat{y}}{\partial z^{[2]}}$ is the model's prediction with respect to the second layer's **linear function** i.e., $Z = WX + b$

- $\frac{\partial z^{[2]}}{\partial a^{[1]}_1}$ is the second layer's linear function with respect to the first layer's activation $a^{[1]}_1$

- $\frac{\partial a^{[1]}_1}{\partial z^{[1]}_1}$ is the first layer's activation with respect to the first layer's linear function.

- $\frac{\partial z^{[1]}_1}{\partial W^{[1]}_1}$ is the first layer's linear function with respect to the first weight node of the first layer $W^{[1]}_1$.


#### Calculate Each Derivative From The Chain Formula Above

Term 1: $$ \frac{\partial L}{\partial \hat{y}} $$

In [8]:
dL_dy_hat = (y_hat.item() - ground_truth.item()) / (y_hat.item() *(1 - y_hat.item()))
print(f"Term 1 (dL/dy_hat): {dL_dy_hat:.4f}")

Term 1 (dL/dy_hat): -1.5117


Term 2: $$\frac{\partial \hat{y}}{\partial z^{[2]}}$$

"How does the sigmoid's output $(\hat y)$ change when its **input** $(z^{[2]})$ changes."

How to get the derivative of the sigmoid function:

- The sigmoid function formula is:
$$sigmoid(z) = \frac{1}{1 + e^{-z} }$$
- This is the function that is calculated in the model's forward() ↓
    ```Python
    class Net()
        ...
        def forward():
            ...
            a2 = F.sigmoid(z2) # ← here
            ...
    ```

The Derivative of the sigmoid function formula is:
$$\sigma'(z) = \sigma(z) * (1- \sigma(z))$$
* Which we use below

In [9]:
dy_hat_dz2 = y_hat.item() * (1 - y_hat.item())
print(f"Term 2 (dy/hat_dz2): {dy_hat_dz2:.4f}")

Term 2 (dy/hat_dz2): 0.2239


Term 3: $$\frac{\partial z^{[2]}}{\partial a^{[1]}_1}$$

The derivative with respect to $a^{[1]}_1$ is just the corresponding weight $W^{[2]}_1$.

In [10]:
W2 = model.fc2.weight
dz2_da1_1 = W2[0,0].item() # The weight connecting a1_1 to z2
print(f"Term 3 (dz2/da1_1): {dz2_da1_1:.4f} (This is W[2]_1), the weight connecting a[1]_1 to z[2]")

Term 3 (dz2/da1_1): 0.3000 (This is W[2]_1), the weight connecting a[1]_1 to z[2]


Term 4: $$\frac{\partial a^{[1]}_1}{\partial z^{[1]}_1}$$

In [11]:
# Formula: 1 if input > 0, else 0
z1_1 = z1[0,0].item() # The first neuron's pre-activation
da1_1_dz1_1 = 1.0 if z1_1 >0 else 0.0
print(f"Term 4 (da1_1_dz1_1): {da1_1_dz1_1:.4f} (z1_1 was {z1_1:.4f})")

Term 4 (da1_1_dz1_1): 1.0000 (z1_1 was 0.5500)


Term 5: $$\frac{\partial z^{[1]}_1}{\partial W^{[1]}_1}$$

The derivative with respect to $W^{[1]}_1$ is just the input $X$ (the positive or negative number)

In [12]:
dz1_1_dW1_1 = input_num.item()
print(f"Term 5 (dz1_1_dW1_1): {dz1_1_dW1_1:.4f}")

Term 5 (dz1_1_dW1_1): 0.5000


#### Calculate the Chain Rule (Reaction)

$$\frac{\partial L}{\partial W^{[1]}_1}=...$$

The final gradient for $W^{[1]}_1$

In [13]:
dL_dW1_1 = dL_dy_hat * dy_hat_dz2 * dz2_da1_1 * da1_1_dz1_1 * dz1_1_dW1_1
print(f"Gradient for W[1]_1: {dL_dW1_1:.8f}")

Gradient for W[1]_1: -0.05077452


Verify if my calculations are the same as Pytorch's

In [14]:
model.zero_grad() # clear old gradients
loss.backward() # Perform backpropagation with built-in Pytorch, i.e all the derivative formulas we did above

# Get the gradient Pytorch calculated for W[1]_1
pytorch_grad = model.fc1.weight.grad[0,0].item()

print(f"Pytorch gradient for W[1]_1: {pytorch_grad:.8f}")
print(f"My manually calculated gradient for W[1]_1: {dL_dW1_1:.8f}\n")

# Compare
if np.isclose(dL_dW1_1, pytorch_grad):
    print("✅ Calculations done correctly!")
else:
    print("❌ Mismatch: Something is wrong in my manually Chain Rule calculations!")

Pytorch gradient for W[1]_1: -0.05077452
My manually calculated gradient for W[1]_1: -0.05077452

✅ Calculations done correctly!


#### What do we do with the gradient for $W^{[1]}_1$

⭐️ We use the gradient for $W^{[1]}_1$ to alter $W^{[1]}_1$, so that the model performs better. We do this for every weight and bias node in the network, each one having their **own corresponding gradient**.

The gradient tells us two things:
1. **The direction**: The **sign** (positive or negative) of the gradient tells you the 'direction of steepest ascent' (i.e., which way is 'uphill' on the Loss landscape). Since our goal is gradient descent, we want to move in the opposite direction (i.e., move in the direction of the 'steepest' descent) to lower the Loss error.

   - If the gradient is **positive** we need to **decrease** the value of $W^{[1]}_1$, i.e., on the Loss landscape we need to move to the **opposite direction** of the sign.

   - If it's **negative** this means that increasing $W^{[1]}_1$ will decrease the Loss making the model perform better. So on the Loss landscape we need to move in the **same direction**, i.e., **increase** $W^{[1]}_1$.
2. **The Magnitude**: The size of the gradient (eg., $0.8$ vs $0.0001$) tells us how *steep* the Loss is. A larger value means the Loss is very sensitive to this weight, and a small change will have a big impact.

#### How Much Do We Change The Value Of $W^{[1]}_1$

Given the gradient of $W^{[1]}_1$, we take a small step in the correct direction on the **Loss landscape**. The step size is controlled by the hyper-parameter **Learning Rate** (often called $\alpha$ "alpha"). This learning rate is a small arbitrary float (e.g., 0.01, or 0.0001, etc...).

There are many algorithms that we can use to update $W^{[1]}_1$, the below example is the **Gradient Descent** algorithm:

$$ W^{[1]}_{new} = W^{[1]}_{old} - (\alpha * gradient) $$

**Example Cases**:

-   The old value of $W^{[1]}_1$ is $0.29$
-   The learning rate is $0.01$

1. Case 1: **Gradient is positive** (e.g., $0.3$)
    -   $ W^{[1]}_{new} = 0.29 - (0.01 * 0.3) $
    -   The weight decreases.
2. Case 2: **Gradient is negative** (e.g., $-0.4$)
    -   $ W^{[1]}_{new} = 0.29 - (0.01 * -0.4) $
    -   $ W^{[1]}_{new} = 0.29 + 0.004$
    -   The weight increases.

So the gradient $\frac{\partial L}{\partial W^{[1]}_1}$ doesn't tell you the exact amount to change the $W^{[1]}_1$, but gives you the **direction** and **relative amount** to "nudge" the weight to improve the model, one small step at a time.

In [15]:
def updateSingleWeight(learning_rate, gradient, old_value):
    """Update a single weight node."""
    return old_value - (learning_rate * gradient)

In [16]:
learning_rate = 0.1

print(f"Old value of W[1]_1: {model.fc1.weight[0,0].item()}") # First weight of the first layer.
print(f"The gradient of W[1]_1: {dL_dW1_1}")

Old value of W[1]_1: 0.8999999761581421
The gradient of W[1]_1: -0.050774522653203036


Update the value of $W^{[1]}_1$

In [17]:
with torch.no_grad():
    model.fc1.weight[0,0] = updateSingleWeight(learning_rate, dL_dW1_1, model.fc1.weight[0,0].item())

print(f"New value of W[1]_1: {model.fc1.weight[0,0]}")

New value of W[1]_1: 0.9050774574279785


Predict again with a new value for $W^{[1]}_1$:

In [18]:
# Prediction
y_hat, _, _, _ = model(input_num)
new_prediction = y_hat

# Loss
new_loss = criterion(new_prediction, ground_truth)

print(f"Old loss: {loss:.6f} | New Loss: {new_loss:.6f}")

Old loss: 0.413240 | New Loss: 0.412983


💡 As we can see the new Loss is closer to 0! Meaning that by 'nudging' $W^{[1]}_1$ in the correct direction has made the model better. **Now we need to do the same for every other weight and bias node in the network by their own corresponding gradient!**